In [1]:
from mdcrow import MDCrow
from langchain.callbacks import get_openai_callback
from datetime import datetime
import os 
import traceback

In [2]:
import sys
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '../'))
sys.path.append(parent_dir)
from robustness_prompts import get_prompt # noqa: E402

prompt_9_natural = get_prompt("natural", 9)

prompt_9_natural

'Simulate 1LYZ for 1ps at 300 K and for 1ps at 400K. Report the secondary structure assignments of the downloaded PDB structure and any known binding or active sites. Compute SASA and radius of gyration of the 300K simulation. Compute RMSD of both simulations.'

In [3]:
llm_model = "gpt-4-turbo-2024-04-09"
tools = "all"

In [4]:
agent = MDCrow(
    agent_type="Structured", 
    model=llm_model, 
    top_k_tools=tools, 
    use_memory=False,
    streaming=False,
    verbose=True,
)
with get_openai_callback() as cb:
    chat_start = datetime.now()
    try:
        response = agent.run(prompt_9_natural, callbacks=[cb])
        print(response)
    except Exception as e:
        exc_type, exc_value, exc_traceback = sys.exc_info()
        print(f'{type(e).__name__}:{e}')
        print("".join(traceback.format_exception(exc_type, exc_value, exc_traceback)))
    chat_end = datetime.now()
    total_runtime = (chat_end - chat_start).total_seconds()
    print(cb)
    print(f"Total runtime: {total_runtime:.2f}s")



> Entering new AgentExecutor chain...


Thought: To address the question, I need to perform several tasks: simulate the protein 1LYZ at two different temperatures, analyze its secondary structure, identify any known binding or active sites, compute SASA and radius of gyration for the 300K simulation, and compute RMSD for both simulations. First, I will download the PDB structure of 1LYZ and then proceed with the simulations.

Action:
```json
{
  "action": "PDBFileDownloader",
  "action_input": {
    "query": "1LYZ"
  }
}
```PDB file found with this ID: 1LYZ

Observation: Succeeded. Downloaded the PDB file:1LYZ_165223
Thought:

Now that I have the PDB file for 1LYZ, I will proceed to set up and run the molecular dynamics simulations at 300K and 400K. I will use the downloaded PDB file as the input for these simulations.

Action:
```json
{
  "action": "SetUpandRunFunction",
  "action_input": {
    "pdb_id": "1LYZ_165223",
    "forcefield_files": ["amber99sb-ildn"],
    "system_params": {
      "nonbondedMethod": "PME",
      "nonbondedCutoff": "1.0 * nanometers",
      "constraints": "HBonds",
      "rigidWater": true,
      "solvate": true
    },
    "integrator_params": {
      "integrator_type": "LangevinMiddle",
      "Temperature": "300 * kelvin",
      "Friction": "1.0 / picoseconds",
      "Timestep": "0.002 * picoseconds"
    },
    "simulation_params": {
      "Ensemble": "NVT",
      "Number of Steps": 500,
      "record_interval_steps": 5
    }
  }
}
```Setting default ewaldErrorTolerance: 0.0005 
error found: constraintTolerance must be specified if rigidWater is TrueThe forcefield file is not pres

To address the simulation setup failure, I need to correct the forcefield file name and include the constraintTolerance parameter. I will use the available forcefield 'amber99sbildn.xml' and specify a constraintTolerance. I will then rerun the simulation setup for 300K and 400K.

Action:
```json
{
  "action": "SetUpandRunFunction",
  "action_input": {
    "pdb_id": "1LYZ_165223",
    "forcefield_files": ["amber99sbildn.xml"],
    "system_params": {
      "nonbondedMethod": "PME",
      "nonbondedCutoff": "1.0 * nanometers",
      "constraints": "HBonds",
      "rigidWater": true,
      "solvate": true,
      "constraintTolerance": "0.00001"
    },
    "integrator_params": {
      "integrator_type": "LangevinMiddle",
      "Temperature": "300 * kelvin",
      "Friction": "1.0 / picoseconds",
      "Timestep": "0.002 * picoseconds"
    },
    "simulation_params": {
      "Ensemble": "NVT",
      "Number of Steps": 500,
      "record_interval_steps": 5
    }
  }
}
```Setting default ewald

About to create system...
Attempts at creating system: 0/3
Error adding solvent ValueError – No template found for residue 1 (LYS).  The set of atoms is similar to HID, but it is missing 8 atoms.  For more information, see https://github.com/openmm/openmm/wiki/Frequently-Asked-Questions#template
Residue code: LYS
Residue code not in solvent list. Adding forcefield                         not supported.

Observation: Failed. No template found for residue 1 (LYS).  The set of atoms is similar to HID, but it is missing 8 atoms.  For more information, see https://github.com/openmm/openmm/wiki/Frequently-Asked-Questions#templateThis were the inputs {'pdb_id': '1LYZ_165223', 'forcefield_files': ['amber99sbildn.xml'], 'system_params': {'nonbondedMethod': 'PME', 'nonbondedCutoff': '1.0 * nanometers', 'constraints': 'HBonds', 'rigidWater': True, 'solvate': True, 'constraintTolerance': '0.00001'}, 'integrator_params': {'integrator_type': 'LangevinMiddle', 'Temperature': '300 * kelvin', 'Friction

...

To address the issue with the missing template for residue HOH, it appears that the water molecules were not properly handled during the cleaning process. I will re-clean the PDB file, ensuring that water molecules are properly removed to avoid issues with the forcefield compatibility during the simulation setup.

Action:
```json
{
  "action": "CleaningToolFunction",
  "action_input": {
    "pdb_id": "1LYZ_165342",
    "replace_nonstandard_residues": true,
    "add_missing_atoms": true,
    "remove_heterogens": true,
    "remove_water": true,
    "add_hydrogens": true,
    "add_hydrogens_ph": 7.0
  }
}
```file path: /gpfs/fs2/scratch/qcampbe2/repos/MDCrow/ckpt/ckpt_39/pdb/1LYZ_Clean_20251223_165342.pdb



Observation: Succeeded. File cleaned! 
File ID: 1LYZ_165405
Thought:

Now that the PDB file has been successfully cleaned again, ensuring that water molecules are properly removed, I will proceed to set up and run the molecular dynamics simulations at 300K and 400K using the cleaned PDB file. I will adjust the simulation parameters to ensure compatibility with the forcefield and the cleaned file.

Action:
```json
{
  "action": "SetUpandRunFunction",
  "action_input": {
    "pdb_id": "1LYZ_165405",
    "forcefield_files": ["amber99sbildn.xml"],
    "system_params": {
      "nonbondedMethod": "PME",
      "nonbondedCutoff": "1.0 * nanometers",
      "constraints": "HBonds",
      "rigidWater": true,
      "solvate": true,
      "constraintTolerance": "0.00001"
    },
    "integrator_params": {
      "integrator_type": "LangevinMiddle",
      "Temperature": "300 * kelvin",
      "Friction": "1.0 / picoseconds",
      "Timestep": "0.002 * picoseconds"
    },
    "simulation_params": {
      "Ensemble": "NVT",
      "Number of Steps": 500,
      "record_inter

Error adding solvent ValueError – No template found for residue 130 (HOH).  This might mean your input topology is missing some atoms or bonds, or possibly that you are using the wrong force field.  For more information, see https://github.com/openmm/openmm/wiki/Frequently-Asked-Questions#template
Residue code: HOH
Trying to add missing component to Forcefield...
No SMILES found for HET code.

Observation: Failed. No template found for residue 130 (HOH).  This might mean your input topology is missing some atoms or bonds, or possibly that you are using the wrong force field.  For more information, see https://github.com/openmm/openmm/wiki/Frequently-Asked-Questions#templateThis were the inputs {'pdb_id': '1LYZ_165405', 'forcefield_files': ['amber99sbildn.xml'], 'system_params': {'nonbondedMethod': 'PME', 'nonbondedCutoff': '1.0 * nanometers', 'constraints': 'HBonds', 'rigidWater': True, 'solvate': True, 'constraintTolerance': '0.00001'}, 'integrator_params': {'integrator_type': 'Langev

Final Answer: After multiple attempts to clean and prepare the PDB file of the protein 1LYZ for simulation, and despite repeated efforts to address issues with missing templates for water molecules, the simulation setup has consistently failed. The repeated issue pertains to the absence of a template for residue HOH (water), indicating a persistent problem with the handling of water molecules in the context of the selected forcefield and simulation parameters. This suggests a deeper compatibility issue between the cleaned PDB file and the forcefield or an error in the cleaning process that did not effectively remove or appropriately handle water molecules. Further investigation and possibly alternative approaches or tools might be required to resolve this issue and successfully run the simulations.

> Finished chain.
({'input': '\n    You are an expert molecular dynamics scientist, and\n    your task is to respond to the question or\n    solve the problem to the best of your ability us

In [3]:
registry = agent.path_registry
print(registry.list_path_names_and_descriptions().replace(",", "\n"))

Files found in registry: 1LYZ_165223: PDB file downloaded from RSCB
 PDBFile ID: 1LYZ_165223
 1LYZ_165248: Cleaned File:  Removed Heterogens
 and Water Removed.  Replaced Nonstandard Residues. Added Hydrogens at pH 7.0. Missing Atoms added and nonstandard residues replaced. 
 1LYZ_165304: Cleaned File:  Removed Heterogens
 and Water Removed.  Replaced Nonstandard Residues. Added Hydrogens at pH 7.0. Missing Atoms added and nonstandard residues replaced. 
 1LYZ_165320: Cleaned File:  Removed Heterogens
 and Water Removed.  Replaced Nonstandard Residues. Added Hydrogens at pH 7.0. Missing Atoms added and nonstandard residues replaced. 
 1LYZ_165342: Cleaned File:  Removed Heterogens
 and Water Removed.  Replaced Nonstandard Residues. Added Hydrogens at pH 7.0. Missing Atoms added and nonstandard residues replaced. 
 1LYZ_165405: Cleaned File:  Removed Heterogens
 and Water Removed.  Replaced Nonstandard Residues. Added Hydrogens at pH 7.0. Missing Atoms added and nonstandard residues rep

In [4]:
# make sure pdb was downloaded
assert os.path.exists(registry.get_mapped_path("1LYZ_165223"))

In [5]:
# # make sure dssp was computed correctly
# from mdcrow.tools.base_tools import ComputeDSSP

# dssp = ComputeDSSP(registry)
# dssp._run(traj_file= "1LYZ_125306", target_frames="first")

In [6]:
# # make sure the sites were found
# from mdcrow.tools.base_tools import GetAllKnownSites

# get_all_known_sites = GetAllKnownSites()
# get_all_known_sites._run(query="1LYZ", primary_accession="")

In [7]:
# # make sure trajectory and topology exist
# traj_path_1 = registry.get_mapped_path("rec0_125315")
# top_path_1 = registry.get_mapped_path("top_sim0_023025")

# assert os.path.exists(traj_path_1)
# assert os.path.exists(top_path_1)

In [8]:
# # make sure rmsd plot was generated
# from IPython.display import Image
# Image(filename=registry.get_mapped_path('fig0_022913'))

In [9]:
# # make sure rgy plot was generated
# from IPython.display import Image
# Image(filename=registry.get_mapped_path('fig0_023033'))

In [10]:
# # make sure sasa plot was generated
# from IPython.display import Image
# Image(filename=registry.get_mapped_path('fig0_023320'))

In [11]:
# # make sure trajectory and topology exist (sim2)
# traj_path_2 = registry.get_mapped_path("rec0_023742")
# top_path_2 = registry.get_mapped_path("top_sim0_023742")

# assert os.path.exists(traj_path_2)
# assert os.path.exists(top_path_2)

In [12]:
# # make sure rmsd plot was generated (sim2)
# from IPython.display import Image
# Image(filename=registry.get_mapped_path('fig0_024812'))

In [16]:
#verify the total cost
def calculate_llm_cost(input_tokens, output_tokens, model):
    pricing_2024 = {
        "gpt-4-1106-preview": {"input": 10/1e6, "output": 30/1e6},
        "gpt-3.5-turbo-0125": {"input": 0.5/1e6, "output": 1.5/1e6},
        "gpt-4-turbo-2024-04-09": {"input": 10/1e6, "output": 30/1e6},
        "gpt-4o-2024-08-06": {"input": 5/1e6, "output": 15/1e6},
        "llama-v3p1-70b-instruct": {"input": 0.9/1e6, "output": 0.9/1e6},
        "llama-v3p1-405b-instruct": {"input": 3/1e6, "output": 3/1e6},
        "claude-3-opus": {"input": 15/1e6, "output": 75/1e6},
        "claude-3.5-sonnet": {"input": 3/1e6, "output": 15/1e6},
    }
    pricing_2025 = {
        "gpt-4-1106-preview": {"input": 10/1e6, "output": 30/1e6},
        "gpt-3.5-turbo-0125": {"input": 0.5/1e6, "output": 1.5/1e6},
        "gpt-4-turbo-2024-04-09": {"input": 10/1e6, "output": 30/1e6},
        "gpt-4o-2024-08-06": {"input": 2.5/1e6, "output": 10/1e6},
        "llama-v3p1-70b-instruct": {"input": 0.9/1e6, "output": 0.9/1e6},
        "llama-v3p1-405b-instruct": {"input": 3/1e6, "output": 3/1e6},
    }
    
    prices = pricing_2025[model]
    cost = (input_tokens * prices["input"]) + (output_tokens * prices["output"])
    return round(cost, 6)

llm_cost = calculate_llm_cost(cb.prompt_tokens, cb.completion_tokens, agent.llm.model_name)
print('Input tokens:',cb.prompt_tokens)
print('Output tokens:',cb.completion_tokens)
print('LLM costs: $',llm_cost)

Input tokens: 141933
Output tokens: 2960
LLM costs: $ 1.50813
